# YOLO11 -> TFLite export

Convert an Ultralytics `.pt` weight to **TFLite**. You **cannot** convert the NCNN
folder directly — always export from the original `.pt`.

Export pipeline (handled by Ultralytics): `PyTorch -> ONNX -> TF SavedModel -> TFLite` (via `onnx2tf`).

Outputs land in `<model>_saved_model/`:
- `<model>_float32.tflite`
- `<model>_float16.tflite`  (smaller, recommended for CPU/edge)
- `<model>_int8.tflite`     (only if `INT8=True` + calibration data)

Runs locally (the `.pt` files are in `../scripts/ai_models/`) or on Kaggle/Colab.
Set `IMGSZ` to the size you run inference at — in `base_reader.py` that is **640**.

In [ ]:
# 1. Dependencies
# Ultralytics auto-installs most TFLite deps during export, but pre-installing
# avoids version-resolution surprises mid-export.
!pip install -U ultralytics -q
!pip install -q onnx onnxslim onnx_graphsurgeon sng4onnx onnx2tf "tensorflow>=2.0.0"

import ultralytics, torch

# Patch for PyTorch >=2.6 (weights_only default changed) so custom .pt files load.
_orig_load = torch.load
def _patched_load(f, map_location=None, pickle_module=None, **kwargs):
    kwargs['weights_only'] = False
    if pickle_module is not None:
        return _orig_load(f, map_location=map_location, pickle_module=pickle_module, **kwargs)
    return _orig_load(f, map_location=map_location, **kwargs)
torch.load = _patched_load

print('ultralytics:', ultralytics.__version__)
print('torch:', torch.__version__)

In [ ]:
# 2. Config
from pathlib import Path

# Folder with the .pt weights (relative to this notebook in notebooks/).
MODELS_DIR = Path('../scripts/ai_models')

# Which weight to convert. Pick one of the .pt below.
MODEL_PATH = MODELS_DIR / 'pesocomcameradrone.pt'

# Inference image size. Must match what you run at runtime (base_reader.py uses 640).
IMGSZ = 640

# float16 is the recommended default for CPU/edge (half size, ~same accuracy).
HALF = True

# int8 needs a representative calibration dataset (a data.yaml with val images).
# Leave INT8=False unless you have one — int8 can hurt keypoint accuracy on pose models.
INT8 = False
CALIB_DATA = None   # e.g. Path('../path/to/data.yaml')

print('Available .pt weights:')
for p in sorted(MODELS_DIR.glob('*.pt')):
    print('  ', p.name, f'({p.stat().st_size/1e6:.1f} MB)')
assert MODEL_PATH.exists(), f'Not found: {MODEL_PATH.resolve()}'
print('\nSelected:', MODEL_PATH.resolve())

In [ ]:
# 3. Load and inspect (task is auto-detected: pose / detect / ...)
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
print('task:', model.task)
print('names:', model.names)
if model.task == 'pose':
    print('kpt_shape:', model.model.kpt_shape)

In [ ]:
# 4. Export to TFLite
# Produces <model>_saved_model/<model>_float32.tflite and _float16.tflite.
# With INT8=True it also writes _int8.tflite (requires CALIB_DATA).
export_kwargs = dict(format='tflite', imgsz=IMGSZ, half=HALF)
if INT8:
    assert CALIB_DATA and Path(CALIB_DATA).exists(), 'INT8 needs a valid CALIB_DATA data.yaml'
    export_kwargs.update(int8=True, half=False, data=str(CALIB_DATA))

exported = model.export(**export_kwargs)
print('\nExport returned:', exported)

In [ ]:
# 5. Locate the produced .tflite files
saved_dir = MODEL_PATH.with_name(MODEL_PATH.stem + '_saved_model')
tflites = sorted(saved_dir.glob('*.tflite'))
print('Output dir:', saved_dir.resolve())
for t in tflites:
    print(f'  {t.name:32s} {t.stat().st_size/1e6:6.2f} MB')

# Prefer fp16 for runtime if present, else the first available.
TFLITE_PATH = next((t for t in tflites if 'float16' in t.name), tflites[0])
print('\nUsing for validation:', TFLITE_PATH.name)

In [ ]:
# 6. Validate: compare .pt vs .tflite on a real image
# Set IMG_PATH to a test frame. If left None, a synthetic image is used just to
# confirm the model loads/runs (keypoints won't be meaningful in that case).
import numpy as np

IMG_PATH = None   # e.g. '../some_test_frame.jpg'

if IMG_PATH:
    source = str(IMG_PATH)
else:
    source = np.random.randint(0, 255, (IMGSZ, IMGSZ, 3), dtype=np.uint8)
    print('No IMG_PATH set — using a synthetic image (sanity check only).')

model_tf = YOLO(str(TFLITE_PATH), task=model.task)

r_pt = model(source, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
r_tf = model_tf(source, imgsz=IMGSZ, conf=0.25, verbose=False)[0]

print(f'detections   pt={len(r_pt.boxes)}  tflite={len(r_tf.boxes)}')
if model.task == 'pose' and len(r_pt.boxes) and len(r_tf.boxes):
    kp_pt = r_pt.keypoints.xy[0].cpu().numpy()
    kp_tf = r_tf.keypoints.xy[0].cpu().numpy()
    err = np.linalg.norm(kp_pt - kp_tf, axis=1)
    print('keypoint px error (pt vs tflite):')
    print('  per-kp:', np.round(err, 2))
    print(f'  mean={err.mean():.2f}px  max={err.max():.2f}px')

In [ ]:
# 7. Latency benchmark (CPU) — you care about runtime perf
import time, numpy as np

dummy = np.random.randint(0, 255, (IMGSZ, IMGSZ, 3), dtype=np.uint8)

def bench(m, n=30, warmup=5):
    for _ in range(warmup):
        m(dummy, imgsz=IMGSZ, verbose=False)
    t0 = time.perf_counter()
    for _ in range(n):
        m(dummy, imgsz=IMGSZ, verbose=False)
    return (time.perf_counter() - t0) / n * 1000

ms_tf = bench(model_tf)
print(f'TFLite ({TFLITE_PATH.name}): {ms_tf:.1f} ms/frame  (~{1000/ms_tf:.1f} FPS)')

## 8. Use it in `base_reader.py`

Ultralytics loads a `.tflite` the same way as the NCNN folder — just point the
`~model` param at the `.tflite` file:

```python
# scripts/ai_models/<model>_saved_model/<model>_float16.tflite
model = YOLO(model_path, task='pose')
results = model(frame, imgsz=640, conf=0.8)
```

Copy the chosen `.tflite` into `scripts/ai_models/` (or pass the full path via the
ROS `~model` parameter) and keep `imgsz` identical to what you exported with (640).

Notes:
- **float16** is the best default on CPU/ARM: half the size, accuracy basically unchanged.
- **int8** is faster/smaller but needs calibration data and tends to degrade keypoint
  precision on pose models — validate the px error in cell 6 before shipping it.
- TFLite multi-threading is controlled by Ultralytics; if you need to cap threads on
  the companion computer, set the OpenCV/OMP thread env vars before launching the node.